# Séance 4 — Premier modèle « clé en main »

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ts/notebooks/TS_S4_premier_modele.ipynb)
**Objectif :** entraîner un arbre de décision qui prédit si un intervalle est minéralisé (> 0,5 g/t), lire ses règles et ses erreurs.

> Fichiers nécessaires : téléverser les CSV du dossier `TS_TP_donnees` (icône dossier à gauche dans Colab).
> Exécuter chaque cellule avec ▶ (ou Maj + Entrée).

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['sondages_miniers_propre.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ts/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

df = pd.read_csv('sondages_miniers_propre.csv')
df['quartz'] = (df['lithologie'] == 'Veine de quartz').astype(int)
df['mineralise'] = (df['au_gpt'] > 0.5).astype(int)
print(df['mineralise'].value_counts())

## 1. Séparer : apprendre (75 %) / tester (25 %)

In [ ]:
X = df[['quartz', 'sulfures_pct', 'alteration', 'de_m']]
y = df['mineralise']
X_app, X_test, y_app, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

## 2. Entraîner l’arbre
### À MODIFIER : `max_depth` = 2, puis 3, 5, 10

In [ ]:
arbre = DecisionTreeClassifier(max_depth=2, random_state=0)
arbre.fit(X_app, y_app)
print(export_text(arbre, feature_names=list(X.columns)))
plt.figure(figsize=(12, 5)); plot_tree(arbre, feature_names=list(X.columns), class_names=['Stérile', 'Minéralisé'], filled=True); plt.show()

## 3. Mesurer les erreurs sur le test

In [ ]:
pred = arbre.predict(X_test)
vn, fp, fn, vp = confusion_matrix(y_test, pred).ravel()
print('Bonnes réponses :', round(accuracy_score(y_test, pred) * 100), '%')
print('VP =', vp, ' FP =', fp, ' FN =', fn, ' VN =', vn)
print('Minéralisés retrouvés :', round(vp / (vp + fn) * 100), '%')

## 4. Tableau à remplir
| max_depth | Bonnes réponses (%) | FN | Minéralisés retrouvés (%) |
|---|---|---|---|
| 2 | | | |
| 3 | | | |
| 5 | | | |
| 10 | | | |

**Questions :** quelle est la première question de l’arbre ? Un géologue serait-il d’accord ? Le meilleur `max_depth` est-il le plus grand ?